In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from pathlib import Path
import os

In [2]:
data_pipeline = "no-feature-eng"
dataset_type = "features"

In [3]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [4]:
raw_train_id = pd.read_csv(f"{data_path}/raw/train.csv")["id"]
X = pd.read_csv(Path(data_path) / Path(data_pipeline) / f"train_{dataset_type}.csv")
X_test = pd.read_csv(Path(data_path) / Path(data_pipeline) / f"test_{dataset_type}.csv")
y = pd.read_csv(Path(data_path) / Path(data_pipeline) / "train_labels.csv")

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   age                      662440 non-null  float64
 1   daily_screen_time_hours  595515 non-null  float64
 2   social_media_hours       557374 non-null  float64
 3   gaming_hours             564548 non-null  float64
 4   work_study_hours         639851 non-null  float64
 5   sleep_hours              646889 non-null  float64
 6   notifications_per_day    623785 non-null  float64
 7   app_opens_per_day        610659 non-null  float64
 8   weekend_screen_time      579306 non-null  float64
 9   gender                   662335 non-null  str    
 10  stress_level             636221 non-null  float64
 11  academic_work_impact     647145 non-null  float64
dtypes: float64(11), str(1)
memory usage: 63.3 MB


In [5]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [6]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        frame[col] = frame[col].astype('category')

In [9]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)
TE_cols = [f"{col}_TE" for col in X.columns]
X_te = pd.DataFrame(index=X.index, columns=TE_cols, dtype=float)

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    te = TargetEncoder(smooth="auto")
    te.fit(X_train, y_train)
    X_te.iloc[valid_index] = te.transform(X_valid)

X_te = pd.concat([X, X_te], axis=1)

te = TargetEncoder(smooth="auto")
te.fit(X, y)
X_test_te = pd.DataFrame(
    te.transform(X_test),
    columns=TE_cols,
    index=X_test.index
)
X_test_te = pd.concat([X_test, X_test_te], axis=1)
X_te.info()

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   age                         662440 non-null  float64 
 1   daily_screen_time_hours     595515 non-null  float64 
 2   social_media_hours          557374 non-null  float64 
 3   gaming_hours                564548 non-null  float64 
 4   work_study_hours            639851 non-null  float64 
 5   sleep_hours                 646889 non-null  float64 
 6   notifications_per_day       623785 non-null  float64 
 7   app_opens_per_day           610659 non-null  float64 
 8   weekend_screen_time         579306 non-null  float64 
 9   gender                      662335 non-null  category
 10  stress_level                636221 non-null  float64 
 11  academic_work_impact        647145 non-null  float64 
 12  age_TE                      691369 non-null  float64 
 13  daily_scre

In [8]:
X_te.to_csv(Path(data_path) / Path(data_pipeline) / f"train_{dataset_type}_TE.csv", index=False)
X_test_te.to_csv(Path(data_path) / Path(data_pipeline) / f"test_{dataset_type}_TE.csv", index=False)